# z004 - Receta del profesor (1 modelo)

**Objetivo:** medir **cuánto tarda** y **cuánta ganancia da** la receta del profesor, para elegir Plan A o B.

- Variables: originales + lags y deltas de orden 1 y 2 (corrida C de z003).
- Train 202104, valid 202106, semilla 501419. Referencia: C con la receta vieja = **350,2 M** (misma semilla), 347 M promedio de 5 semillas (B).
- Fijos del profesor: learning_rate 0.005, min_data_in_leaf 0, feature_fraction 0.5, feature_fraction_bynode 0.2, max_bin 31, métrica AUC.
- Los que él optimiza con Optuna acá van con valores razonables: num_leaves 64, min_sum_hessian_in_leaf 1, hasta 3000 árboles con early stopping.

Ojo: el early stopping mira 202106 y la ganancia también se mide en 202106 → el número es algo optimista.

In [1]:
# --- 1. Librerías y parámetros ---
import time
import numpy as np
import pandas as pd
import lightgbm as lgb

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
MESES_TRAIN = [202104]
MES_VALID   = 202106
SEMILLA     = 501419

params = {
    "objective": "binary",
    "metric": "auc",                  # el profesor usa AUC
    "learning_rate": 0.005,           # muy bajo: muchos árboles, mejor generalización
    "min_data_in_leaf": 0,            # sin mínimo de clientes por hoja...
    "min_sum_hessian_in_leaf": 1.0,   # ...lo controla este (Optuna lo optimiza)
    "num_leaves": 64,                 # Optuna lo optimiza
    "feature_fraction": 0.5,          # cada árbol ve el 50% de las variables
    "feature_fraction_bynode": 0.2,   # cada corte ve el 20% de esas
    "max_bin": 31,                    # menos "cajitas" por variable: más rápido
    "seed": SEMILLA,
    "verbose": -1,
}
MAX_ARBOLES = 3000

In [2]:
# --- 2. Cargar datos, target y lags (igual que z003) ---
df = pd.read_parquet("../competencia_01.parquet")
df["clase_binaria"] = (df["clase_ternaria"] == "BAJA+2").astype(int)

no_usar = ["foto_mes", "clase_ternaria", "clase_binaria"]
originales = [c for c in df.columns if c not in no_usar]
cols = [c for c in originales if c != "numero_de_cliente"]

df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
grupo = df.groupby("numero_de_cliente")[cols]
nuevas = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    nuevas += [lag, delta]
df = pd.concat([df] + nuevas, axis=1)
del nuevas, lag, delta, grupo

variables = [c for c in df.columns if c not in no_usar]
train = df[df["foto_mes"].isin(MESES_TRAIN)]
valid = df[df["foto_mes"] == MES_VALID]
y_valid = valid["clase_binaria"].values
print("Variables:", len(variables))

Variables: 761


In [3]:
# --- 3. Entrenar y medir el tiempo ---
datos_train = lgb.Dataset(train[variables], label=train["clase_binaria"])
datos_valid = lgb.Dataset(valid[variables], label=valid["clase_binaria"], reference=datos_train)

inicio = time.time()
modelo = lgb.train(
    params, datos_train, num_boost_round=MAX_ARBOLES,
    valid_sets=[datos_valid],
    callbacks=[lgb.early_stopping(200, verbose=False),   # corta si el AUC no mejora en 200 árboles
               lgb.log_evaluation(500)],                 # imprime el AUC cada 500 árboles
)
minutos = (time.time() - inicio) / 60
print(f"Tiempo: {minutos:.1f} min | árboles entrenados: {modelo.num_trees()} | mejor iteración: {modelo.best_iteration}")
print("AUC en la mejor iteración:", round(modelo.best_score["valid_0"]["auc"], 4))

[500]	valid_0's auc: 0.896034


Tiempo: 0.5 min | árboles entrenados: 633 | mejor iteración: 633
AUC en la mejor iteración: 0.8974


In [4]:
# --- 4. Ganancia según la cantidad de árboles ---
ganancia_cliente = np.where(y_valid == 1, GANANCIA_ACIERTO, -COSTO_ESTIMULO)

def ganancias(prob):
    orden = np.argsort(prob)[::-1]
    acum = np.cumsum(ganancia_cliente[orden])
    n = int(np.argmax(acum)) + 1
    return {"ganancia_max_M": round(acum[n - 1] / 1e6, 1), "mejor_n": n,
            "top10000_M": round(acum[9_999] / 1e6, 1), "top11500_M": round(acum[11_499] / 1e6, 1)}

filas = []
for k in sorted({500, 1000, 1500, 2000, 2500, 3000, modelo.best_iteration}):
    if k <= modelo.num_trees():
        filas.append({"arboles": k, **ganancias(modelo.predict(valid[variables], num_iteration=k))})
tabla = pd.DataFrame(filas)
tabla

,arboles,ganancia_max_M,mejor_n,top10000_M,top11500_M
0,500,383.3,13183,349.8,369.0
1,633,383.6,13130,357.5,369.0


In [5]:
# --- 5. Anotar en resultados_experimentos.csv ---
ruta_res = "resultados_experimentos.csv"
res = pd.read_csv(ruta_res, sep=";", decimal=",", encoding="utf-8-sig")
res = res[~res["descripcion"].str.startswith("z004")]   # sin duplicar si se vuelve a correr

mejor = tabla.loc[tabla["arboles"] == modelo.best_iteration].iloc[0]
nueva = {
    "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "descripcion": f"z004 receta profesor, lags 1 y 2, best_iter {modelo.best_iteration}, {minutos:.1f} min",
    "meses_train": str(MESES_TRAIN),
    "mes_valid": MES_VALID,
    "semilla": SEMILLA,
    "params": str(params),
    "mejor_n": int(mejor["mejor_n"]),
    "ganancia_max_M": mejor["ganancia_max_M"],
    "ganancia_top10000_M": mejor["top10000_M"],
    "auc": round(modelo.best_score["valid_0"]["auc"], 3),
}
res = pd.concat([res, pd.DataFrame([nueva])], ignore_index=True)
res["n_enviados"] = res["n_enviados"].astype("Int64")
res.to_csv(ruta_res, sep=";", decimal=",", index=False, encoding="utf-8-sig")
res.tail(1)

,fecha,descripcion,meses_train,mes_valid,semilla,params,n_corte_0025,ganancia_0025_M,mejor_n,ganancia_max_M,prob_en_corte,auc,ks,lift,captura,n_enviados,ganancia_publica_M,ganancia_top10000_M,ganancia_top12000_M
12,2026-10-07 18:58,"z004 receta profesor, lags 1 y 2, best_iter 63...",[202104],202106,501419,"{'objective': 'binary', 'metric': 'auc', 'lear...",NaN,NaN,13130.0,383.6,NaN,0.897,NaN,NaN,NaN,<NA>,NaN,357.5,NaN
